# Okoliš AI — Training on RTX PRO 6000 (96GB VRAM)

Run all cells in order. Total training time: ~8-12 hours (~$14-20).

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.0f} GB")

In [ ]:
# Install dependencies
!pip install open3d plyfile pyyaml scipy --quiet

In [ ]:
# Clone repo
!git clone https://github.com/Vile1278/OkolisAI.git /workspace/OkolisAI
%cd /workspace/OkolisAI

## 2. Download Datasets

Datasets se skidaju direktno na VM (brža veza nego upload s tvog računala).

**Toronto3D** (~4GB): automatski se skida u sljedećem cellu.

**SemanticKITTI** (~80GB): trebaš se registrirati na http://www.semantic-kitti.org/dataset.html, dobiti linkove, pa ih zalijepiti u cell ispod. Trebaš skinuti:
- "Download point clouds" (velodyne, ~80GB)
- "Download labels" (~500MB)

In [ ]:
import os

# ============================================================
# Toronto3D — automatic download
# ============================================================
!mkdir -p /workspace/data/Toronto_3D

# Toronto3D is hosted on Zenodo
TORONTO3D_URL = "https://zenodo.org/records/4540999/files"
TILES = ["L001.ply", "L002.ply", "L003.ply", "L004.ply"]

for tile in TILES:
    dest = f"/workspace/data/Toronto_3D/{tile}"
    if not os.path.exists(dest):
        print(f"Downloading {tile}...")
        !wget -q --show-progress -O {dest} {TORONTO3D_URL}/{tile}
    else:
        print(f"{tile} already exists, skipping.")

print("\n=== Toronto3D ===")
!ls -lh /workspace/data/Toronto_3D/*.ply

In [ ]:
# ============================================================
# SemanticKITTI — paste your download links below
# ============================================================
# 1. Go to http://www.semantic-kitti.org/dataset.html
# 2. Register / login
# 3. Copy the download links for:
#    - "point clouds" (velodyne scans)
#    - "labels" (semantic labels)
# 4. Paste them below and run this cell

!mkdir -p /workspace/data/SemanticKITTI

# ---- PASTE YOUR LINKS HERE ----
VELODYNE_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti/data_odometry_velodyne.zip"
LABELS_URL = "https://semantic-kitti.org/assets/data_odometry_labels.zip"
YAML_URL = "https://github.com/PRBonn/semantic-kitti-api/blob/master/config/semantic-kitti.yaml"

# Uncomment and run when you have the links:
!wget -q --show-progress -O /workspace/data/velodyne.zip "$VELODYNE_URL"
!wget -q --show-progress -O /workspace/data/labels.zip "$LABELS_URL"
!wget -q --show-progress -O /workspace/data/SemanticKITTI/semantic-kitti.yaml "$YAML_URL"

!cd /workspace/data/SemanticKITTI && unzip -q /workspace/data/velodyne.zip
!cd /workspace/data/SemanticKITTI && unzip -q /workspace/data/labels.zip

# Verify
!echo "=== SemanticKITTI ==="
!ls /workspace/data/SemanticKITTI/dataset/sequences/00/velodyne/*.bin 2>/dev/null | head -3 || echo "NOT YET DOWNLOADED"
!ls /workspace/data/SemanticKITTI/dataset/sequences/00/labels/*.label 2>/dev/null | head -3 || echo "LABELS NOT YET DOWNLOADED"

## 3. Patch Code for RTX PRO 6000

Remove memory limits and optimize for 96GB VRAM + 90GB RAM.

In [ ]:
%%writefile okolis_ai/datasets/label_maps.py
"""Unified 8-class taxonomy and per-dataset label remapping.

Classes:
    0  unlabeled   - anything not covered (incl. vehicles, persons)
    1  ground      - grass, dirt, terrain
    2  road        - road surface, asphalt
    3  sidewalk    - sidewalk, parking surface
    4  building    - buildings, walls
    5  fence       - fences, barriers
    6  vegetation  - trees, bushes, hedges
    7  pole        - poles, utility lines, traffic signs
"""
from __future__ import annotations
import numpy as np

NUM_CLASSES = 8

CLASS_NAMES = [
    "unlabeled", "ground", "road", "sidewalk",
    "building", "fence", "vegetation", "pole",
]

CLASS_COLORS = [
    [0.50, 0.50, 0.50],   # 0 unlabeled  - gray
    [0.60, 0.40, 0.20],   # 1 ground     - brown
    [0.25, 0.25, 0.25],   # 2 road       - dark gray
    [0.70, 0.70, 0.70],   # 3 sidewalk   - light gray
    [0.90, 0.20, 0.20],   # 4 building   - red
    [0.90, 0.60, 0.10],   # 5 fence      - orange
    [0.10, 0.65, 0.10],   # 6 vegetation - green
    [0.90, 0.90, 0.20],   # 7 pole       - yellow
]

TORONTO3D_MAP = {
    0: 0, 1: 2, 2: 2, 3: 6, 4: 4, 5: 7, 6: 7, 7: 0, 8: 5,
}

SEMKITTI_MAP = {
    0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0,
    9: 2, 10: 3, 11: 3, 12: 1, 13: 4, 14: 5, 15: 6, 16: 6,
    17: 1, 18: 7, 19: 7,
}

BOTANIC_MAP = {0: 0, 1: 1, 2: 6, 3: 4, 4: 0}

S3DIS_MAP = {
    0: 0, 1: 4, 2: 1, 3: 4, 4: 7, 5: 0, 6: 0, 7: 0, 8: 5, 9: 0, 10: 0, 11: 0,
}


def apply_map(labels: np.ndarray, mapping: dict[int, int]) -> np.ndarray:
    out = np.zeros_like(labels, dtype=np.int64)
    for src, dst in mapping.items():
        out[labels == src] = dst
    return out

In [ ]:
%%writefile okolis_ai/datasets/toronto3d.py
"""Toronto-3D loader — no memory limits (90GB RAM available)."""
from __future__ import annotations
from pathlib import Path
import numpy as np
from .merged import PointCloudTileDataset, LoadedScan
from .label_maps import TORONTO3D_MAP, apply_map


def _read_toronto_ply(path: Path) -> LoadedScan:
    from plyfile import PlyData
    ply = PlyData.read(str(path))
    v = ply["vertex"].data

    xyz = np.stack([v["x"], v["y"], v["z"]], axis=1).astype(np.float32)

    rgb = None
    if all(k in v.dtype.names for k in ("red", "green", "blue")):
        rgb = np.stack([v["red"], v["green"], v["blue"]],
                       axis=1).astype(np.float32) / 255.0

    intensity = None
    for key in ("scalar_Intensity", "intensity", "Intensity"):
        if key in v.dtype.names:
            raw = np.asarray(v[key], dtype=np.float32)
            mx = max(float(raw.max()), 1.0)
            intensity = (raw / mx).astype(np.float32)
            break

    label_raw = None
    for key in ("scalar_Label", "label", "class"):
        if key in v.dtype.names:
            label_raw = np.asarray(v[key], dtype=np.int64)
            break
    if label_raw is None:
        raise ValueError(f"No label field found in {path}")

    labels = apply_map(label_raw, TORONTO3D_MAP)
    del ply, v
    return LoadedScan(xyz=xyz, rgb=rgb, intensity=intensity, labels=labels)


class Toronto3DDataset(PointCloudTileDataset):
    def _load_scan(self, path: Path) -> LoadedScan:
        return _read_toronto_ply(path)


def default_split(root):
    root = Path(root)
    tiles = sorted(root.glob("L00*.ply"))
    if not tiles:
        raise FileNotFoundError(f"No L00*.ply tiles under {root}")
    by_name = {p.stem: p for p in tiles}
    train = [by_name[k] for k in ("L001", "L003", "L004") if k in by_name]
    test  = [by_name[k] for k in ("L002",) if k in by_name]
    val   = [by_name.get("L004")] if "L004" in by_name else []
    train = [p for p in train if p.stem != "L004"]
    return {"train": train, "val": val, "test": test}

In [ ]:
%%writefile okolis_ai/datasets/merged.py
"""Abstract dataset base + shared crop/augment logic (optimized for high-RAM servers)."""
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
import numpy as np
from .common import pack_features, height_above_ground_from_labels, modality_dropout

try:
    from torch.utils.data import Dataset
except ImportError:
    Dataset = object


@dataclass
class LoadedScan:
    xyz: np.ndarray
    rgb: Optional[np.ndarray]
    intensity: Optional[np.ndarray]
    labels: np.ndarray


class PointCloudTileDataset(Dataset):
    ground_label = 1

    def __init__(self, scan_paths, crop_points=65536,
                 voxel=0.03, augment=True,
                 modality_dropout=True, seed=None):
        self.paths = list(scan_paths)
        self.crop = crop_points
        self.voxel = voxel
        self.augment = augment
        self.do_mod_drop = modality_dropout
        self.rng = np.random.default_rng(seed)

    def __len__(self): return len(self.paths)

    def _load_scan(self, path):
        raise NotImplementedError

    def _voxel_downsample(self, scan):
        v = self.voxel
        if v <= 0:
            return scan
        xyz = scan.xyz
        grid = np.floor(xyz / v).astype(np.int32)
        gmin = grid.min(axis=0)
        grid -= gmin
        dims = grid.max(axis=0).astype(np.int64) + 1
        total_cells = dims[0] * dims[1] * dims[2]
        if total_cells < 2**31:
            k = ((grid[:, 0].astype(np.int32) * np.int32(dims[1])
                  + grid[:, 1]) * np.int32(dims[2]) + grid[:, 2])
        else:
            k = ((grid[:, 0].astype(np.int64) * dims[1]
                  + grid[:, 1]) * dims[2] + grid[:, 2])
        _, pick = np.unique(k, return_index=True)
        return LoadedScan(
            xyz=scan.xyz[pick],
            rgb=scan.rgb[pick] if scan.rgb is not None else None,
            intensity=scan.intensity[pick] if scan.intensity is not None else None,
            labels=scan.labels[pick])

    def _anchor_crop(self, scan):
        n = len(scan.xyz)
        if n == 0:
            raise ValueError("empty scan")
        if n <= self.crop:
            pad = self.rng.integers(0, n, size=self.crop - n)
            idx = np.concatenate([np.arange(n), pad])
        else:
            anchor = scan.xyz[self.rng.integers(0, n)]
            d2 = ((scan.xyz - anchor) ** 2).sum(axis=1)
            idx = np.argpartition(d2, self.crop)[:self.crop]
        return LoadedScan(
            xyz=scan.xyz[idx],
            rgb=scan.rgb[idx] if scan.rgb is not None else None,
            intensity=scan.intensity[idx] if scan.intensity is not None else None,
            labels=scan.labels[idx])

    def _augment_geo(self, scan):
        xyz = scan.xyz.copy()
        t = self.rng.uniform(-np.pi, np.pi)
        c, s = np.cos(t), np.sin(t)
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
        xyz = xyz @ R.T
        xyz *= self.rng.uniform(0.9, 1.1)
        xyz += self.rng.normal(0.0, 0.01, xyz.shape).astype(np.float32)
        return LoadedScan(xyz=xyz, rgb=scan.rgb, intensity=scan.intensity, labels=scan.labels)

    def __getitem__(self, i):
        scan = self._load_scan(self.paths[i])
        scan = self._voxel_downsample(scan)
        if self.augment:
            scan = self._augment_geo(scan)
        scan = self._anchor_crop(scan)
        h = height_above_ground_from_labels(
            scan.xyz, scan.labels, ground_label=self.ground_label, cell=1.0)
        feats = pack_features(scan.rgb, scan.intensity, h)
        if self.do_mod_drop:
            feats = modality_dropout(feats)
        xyz = scan.xyz - scan.xyz.mean(axis=0)
        return (xyz.astype(np.float32),
                feats.astype(np.float32),
                scan.labels.astype(np.int64))

## 4. Write Config (optimized for 96GB VRAM)

In [ ]:
%%writefile okolis_ai/configs/rtx_pro_6000.yaml
# RTX PRO 6000 — 96GB VRAM, 90GB RAM, 30 CPUs
datasets:
  toronto3d:
    root: /workspace/data/Toronto_3D
    weight: 1.0
  semantickitti:
    root: /workspace/data/SemanticKITTI
    weight: 0.7

crop_points: 131072
voxel: 0.02
augment: true
modality_dropout: true
steps_per_epoch: 1000

batch_size: 32
num_classes: 8
in_feat_dim: 5
lr: 0.003
epochs: 150
out_dir: /workspace/runs/rtx_pro_6000
num_workers: 8

## 5. Verify Datasets

In [ ]:
import yaml
import sys
sys.path.insert(0, '/workspace/OkolisAI')

cfg = yaml.safe_load(open('okolis_ai/configs/rtx_pro_6000.yaml'))

# Test Toronto3D
from okolis_ai.datasets.toronto3d import default_split as t3d_split
t3d = t3d_split(cfg['datasets']['toronto3d']['root'])
print(f"Toronto3D: {len(t3d['train'])} train, {len(t3d['val'])} val")

# Test SemanticKITTI
from okolis_ai.datasets.semantickitti import default_split as sk_split
sk = sk_split(cfg['datasets']['semantickitti']['root'])
print(f"SemanticKITTI: {len(sk['train'])} train, {len(sk['val'])} val")

print("\nAll datasets OK!")

## 6. Train!

In [ ]:
!cd /workspace/OkolisAI && python -m okolis_ai.training.train --config okolis_ai/configs/rtx_pro_6000.yaml

## 7. Download Results

After training, download the best model:

In [ ]:
import os

run_dir = '/workspace/runs/rtx_pro_6000'
best = os.path.join(run_dir, 'best.pt')
last = os.path.join(run_dir, 'last.pt')

if os.path.exists(best):
    size_mb = os.path.getsize(best) / 1e6
    checkpoint = torch.load(best, map_location='cpu', weights_only=False)
    print(f"Best model: {best} ({size_mb:.1f} MB)")
    print(f"  Epoch: {checkpoint.get('epoch', '?')}")
    print(f"  mIoU:  {checkpoint.get('miou', '?')}")
    print(f"\nDownload from JupyterLab file browser: {best}")
else:
    print("Training not complete yet.")